# InceptionV1 mixed4e:55

## Setup

In [2]:
!pip install -q torch-lucent nltk "neural-interp @ git+https://github.com/priyanshujain/neural-interp@main"
import os
if "GTOKEN" not in os.environ:
    from google.colab import auth
    auth.authenticate_user()
    os.environ["GTOKEN"] = !gcloud auth print-access-token
print("token set:", bool(os.environ.get("GTOKEN")))

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
token set: True


In [3]:
!mkdir -p /content/data /content/reference
!test -f /content/inception_pool.tar || curl -sSf -H "Authorization: Bearer $GTOKEN" -o /content/inception_pool.tar "https://storage.googleapis.com/storage/v1/b/pj-okc/o/inception_pool.tar?alt=media"
!test -d /content/data/inception_pool || (tar --warning=no-unknown-keyword -xf /content/inception_pool.tar -C /content/data && find /content/data -name '._*' -delete)
!test -f /content/mixed4e_55_full.tar || curl -sSf -H "Authorization: Bearer $GTOKEN" -o /content/mixed4e_55_full.tar "https://storage.googleapis.com/storage/v1/b/pj-okc/o/mixed4e_55%2Fmixed4e_55_full.tar?alt=media"
!test -d /content/reference/runs || tar -xf /content/mixed4e_55_full.tar -C /content/reference runs/mixed4e_55/space
!ls /content/data/inception_pool | wc -l; ls /content/reference/runs/mixed4e_55/space

123
buckets.json   forest.npz     hits.npz		 trees.npz
distances.npy  hadamards.npy  probabilities.npy


In [4]:
%%time
import json
from concurrent.futures import ThreadPoolExecutor
from functools import cache
from pathlib import Path

import nltk
import numpy as np
import torch
from lucent.modelzoo import inceptionv1
from lucent.modelzoo.inceptionv1.helper_layers import ReluLayer
from nltk.corpus import wordnet
from PIL import Image, ImageDraw
from torch import nn

from neural_interp.buckets import cluster_buckets, set_buckets
from neural_interp.cluster import DensityPeakForest, SoftMembership, collapse_copies
from neural_interp.decomposition import linear_contributions
from neural_interp.layout import PulledSpringLayout, sample_nodes
from neural_interp.metrics import adjusted_rand, composition, weighted_jaccard, weighted_purity
from neural_interp.report import HitSpaceReport, Tile
from neural_interp.selection import ElbowSelector
from neural_interp.space import HitSpace
from neural_interp.torch import Capture, UnitDeepLift

POOL = Path("/content/data/inception_pool")
REFERENCE = Path("/content/reference/runs/mixed4e_55/space")
OUT = Path("/content/runs/same_images")
LAYER, CHANNEL, UPSTREAM = "mixed4e_1x1_pre_relu_conv", 55, "mixed4d"
RESIZE, SIZE, GRID = 256, 224, 14
K, PEAK_SHARE, MIN_BUCKET = 5, 0.8, 10
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
nltk.download("wordnet", quiet=True)
print(DEVICE, OUT)

cuda /content/runs/same_images
CPU times: user 8.68 s, sys: 1.04 s, total: 9.71 s
Wall time: 13.6 s


## Model

In [5]:
%%time
model = inceptionv1(pretrained=True, redirected_ReLU=False)
for name, child in list(model.named_children()):
    if isinstance(child, ReluLayer):
        setattr(model, name, nn.ReLU())
model = model.eval().to(DEVICE)
conv = dict(model.named_modules())[LAYER]
weight = conv.weight[CHANNEL, :, 0, 0].detach().cpu().numpy()
bias = float(conv.bias[CHANNEL].detach())
print(LAYER, CHANNEL, "reads", weight.shape[0], UPSTREAM, "channels, bias", bias)

Downloading: "https://github.com/ProGamerGov/pytorch-old-tensorflow-models/raw/master/inception5h.pth" to /root/.cache/torch/hub/checkpoints/inception5h.pth


100%|██████████| 27.0M/27.0M [00:00<00:00, 154MB/s]


mixed4e_1x1_pre_relu_conv 55 reads 528 mixed4d channels, bias 0.36840683221817017
CPU times: user 148 ms, sys: 242 ms, total: 391 ms
Wall time: 1.5 s


## Images

In [6]:
%%time
def preprocess(path):
    image = Image.open(path).convert("RGB")
    scale = RESIZE / min(image.size)
    image = image.resize((round(image.width * scale), round(image.height * scale)), Image.BILINEAR)
    left, top = (image.width - SIZE) // 2, (image.height - SIZE) // 2
    image = image.crop((left, top, left + SIZE, top + SIZE))
    return torch.from_numpy(np.asarray(image, dtype=np.float32) / 255).permute(2, 0, 1)

def model_input(pixels):
    return pixels * 255 - 117

def load(paths):
    with ThreadPoolExecutor(8) as pool:
        return torch.stack(list(pool.map(preprocess, paths)))

reference_hits = np.load(REFERENCE / "hits.npz", allow_pickle=True)
paths = [POOL / s for s in reference_hits["sample_ids"]]
print(len(paths), "images,", len({p.parent.name for p in paths}), "classes")

2000 images, 123 classes
CPU times: user 258 ms, sys: 0 ns, total: 258 ms
Wall time: 258 ms


## Activations

In [7]:
%%time
chunks = []
with Capture(model, [LAYER]) as capture:
    for start in range(0, len(paths), 64):
        batch = load(paths[start:start + 64]).to(DEVICE)
        chunks.append(capture.run(model_input(batch))[LAYER][:, CHANNEL].cpu().numpy())
activations = np.concatenate(chunks)
flat = activations.ravel()
print(activations.shape, "positive", int((flat > 0).sum()), "max", flat.max())

(2000, 14, 14) positive 176221 max 1561.1606
CPU times: user 19 s, sys: 1.56 s, total: 20.6 s
Wall time: 15.9 s


## Hits

In [8]:
%%time
selector = ElbowSelector(side="positive").fit(flat)
hits = selector.indices_
image_of, y_of, x_of = np.unravel_index(hits, activations.shape)
hit_ids = [f"{paths[i].relative_to(POOL).as_posix()}:{y * GRID + x}" for i, y, x in zip(image_of, y_of, x_of)]
print(len(hits), "hits, cut at", selector.threshold_)

13527 hits, cut at 209.10238647460938
CPU times: user 203 ms, sys: 5.82 ms, total: 209 ms
Wall time: 216 ms


## Hit vectors

In [9]:
%%time
upstream = np.empty((len(hits), weight.shape[0]), dtype=np.float32)
needed = np.unique(image_of)
with Capture(model, [UPSTREAM]) as capture:
    for start in range(0, len(needed), 64):
        ids = needed[start:start + 64]
        batch = load([paths[i] for i in ids]).to(DEVICE)
        maps = capture.run(model_input(batch))[UPSTREAM].cpu().numpy()
        local = {int(i): j for j, i in enumerate(ids)}
        for h in np.flatnonzero(np.isin(image_of, ids)):
            upstream[h] = maps[local[int(image_of[h])], :, y_of[h], x_of[h]]
rows = linear_contributions(upstream, weight)
print(rows.shape, "max |sum - (activation - bias)|", np.abs(rows.sum(1) - (flat[hits] - bias)).max())

(13527, 528) max |sum - (activation - bias)| 0.04940796
CPU times: user 8.9 s, sys: 590 ms, total: 9.49 s
Wall time: 6.19 s


## Distances

In [10]:
%%time
copies = collapse_copies(rows)
distances = weighted_jaccard(copies.points)
print(len(copies.points), "distinct points, max copies", copies.counts.max(), "median distance", np.median(distances))

13527 distinct points, max copies 1 median distance 0.8592639
CPU times: user 7.72 s, sys: 1.55 s, total: 9.27 s
Wall time: 9.3 s


## Forest

In [11]:
%%time
forest = DensityPeakForest(n_neighbors=K).fit(distances, copies.counts)
print(len(forest.roots_), "roots, rho cut", forest.rho_cut_, "delta cut", forest.delta_cut_)

37 roots, rho cut 0.2944130599498749 delta cut 0.5216803550720215
CPU times: user 692 ms, sys: 325 ms, total: 1.02 s
Wall time: 1.09 s


## Soft membership

In [12]:
%%time
membership = SoftMembership(n_neighbors=K, peak_share=PEAK_SHARE).fit(distances, forest)
space = HitSpace(hit_ids, flat[hits], rows, copies.point_of, copies.counts, distances, forest, membership)
print("temperature", membership.temperature_, "confident", int((space.entropy < 0.5).sum()),
      "cluster differs from tree", int((space.cluster != space.tree).sum()))

temperature 0.31935423612594604 confident 3959 cluster differs from tree 4669
CPU times: user 1.23 s, sys: 451 ms, total: 1.68 s
Wall time: 1.69 s


## Save

In [13]:
%%time
space.save(OUT / "space")
np.save(OUT / "space" / "all_activations.npy", flat)
print(sorted(p.name for p in (OUT / "space").iterdir()))

['all_activations.npy', 'distances.npy', 'space.json', 'space.npz', 'summary.json']
CPU times: user 2.83 s, sys: 348 ms, total: 3.18 s
Wall time: 12.1 s


## Buckets and map

In [14]:
%%time
buckets = [b for b in cluster_buckets(space.probabilities) + set_buckets(space.probabilities) if len(b.hits) >= MIN_BUCKET]
hit_distances = distances[np.ix_(space.point, space.point)]
root_hits = np.array([int(np.flatnonzero(space.point == r)[0]) for r in forest.roots_])
nodes = sample_nodes(buckets, space.probabilities, root_hits)
layout = PulledSpringLayout().fit(hit_distances, space.probabilities, nodes, root_hits)
print(len(buckets), "buckets,", len(layout.positions_), "hits on the map")

198 buckets, 1431 hits on the map
CPU times: user 1min 26s, sys: 554 ms, total: 1min 26s
Wall time: 49.4 s


## Report

In [15]:
%%time
GROUPS = {"cat": "feline.n.01", "dog": "dog.n.01", "vehicle": "wheeled_vehicle.n.01"}
COLORS = {"cat": "#f5a623", "dog": "#a0522d", "vehicle": "#4a90e2", "other": "#7f8c8d"}

@cache
def synset(wnid):
    return wordnet.synset_from_pos_and_offset("n", int(wnid[1:]))

@cache
def group(wnid):
    ancestors = {s.name() for path in synset(wnid).hypernym_paths() for s in path}
    return next((g for g, root in GROUPS.items() if root in ancestors), "other")

def wnid(hit_id):
    return hit_id.split("/")[0]

def box(y, x):
    step = SIZE / GRID
    return x * step, y * step, (x + 1) * step, (y + 1) * step

def save_cards(hit, pixels, attribution, y, x):
    array = pixels.numpy().transpose(1, 2, 0)
    image = Image.fromarray((array * 255).astype(np.uint8))
    ImageDraw.Draw(image).rectangle(box(y, x), outline=(255, 214, 0), width=2)
    scale = np.percentile(np.abs(attribution), 99.5) or 1.0
    a = np.clip(attribution / scale, -1, 1)[..., None]
    up = np.array([1.0, 0.85, 0.2]) * np.clip(a, 0, 1)
    down = np.array([0.3, 0.6, 1.0]) * np.clip(-a, 0, 1)
    overlay = Image.fromarray((np.clip(array * 0.35 + up + down, 0, 1) * 255).astype(np.uint8))
    ImageDraw.Draw(overlay).rectangle(box(y, x), outline=(255, 255, 255), width=1)
    image.save(cards / f"hit_{hit}.jpg", quality=88)
    overlay.save(cards / f"hit_{hit}_deeplift.jpg", quality=88)

report = HitSpaceReport(
    space,
    tile=lambda h: tiles[h],
    group_of=lambda h: group(wnid(space.hit_ids[h])),
    group_colors=COLORS,
    all_activations=flat,
    unit="InceptionV1 mixed4e:55",
)
render = report.hits_to_render(buckets)
deeplift = UnitDeepLift(model, LAYER, reduce=lambda a: a.sum(1))
cards = OUT / "report" / "cards"
cards.mkdir(parents=True, exist_ok=True)
tiles = {}
for start in range(0, len(render), 32):
    batch_hits = render[start:start + 32]
    places = [space.hit_ids[h].rsplit(":", 1) for h in batch_hits]
    positions = [divmod(int(pos), GRID) for _, pos in places]
    pixels = load([POOL / path for path, _ in places])
    attributions = deeplift.attribute(
        model_input(pixels.to(DEVICE)),
        model_input(torch.zeros_like(pixels).to(DEVICE)),
        [(CHANNEL, y, x) for y, x in positions],
        (conv.out_channels, GRID, GRID),
    )
    for hit, px, attribution, (y, x) in zip(batch_hits, pixels, attributions, positions):
        save_cards(hit, px, attribution, y, x)
        label = synset(wnid(space.hit_ids[hit])).lemma_names()[0].replace("_", " ")
        tiles[hit] = Tile(f"cards/hit_{hit}.jpg", f"cards/hit_{hit}_deeplift.jpg", label)
print(len(tiles), "cards")

/usr/local/lib/python3.13/dist-packages/captum/log/dummy_log.py:54: UserWarning: Setting forward, backward hooks and attributes on non-linear
               activations. The hooks and attributes will be removed
            after the attribution is finished
  return func(*args, **kwargs)


1881 cards
CPU times: user 44.6 s, sys: 2.82 s, total: 47.4 s
Wall time: 41.5 s


In [16]:
%%time
report.write(OUT / "report", buckets, layout.positions_)
print(len(list((OUT / "report").glob("*.html"))), "pages")

199 pages
CPU times: user 2min 21s, sys: 5.35 s, total: 2min 26s
Wall time: 2min 8s


## Compare with the reference run

In [17]:
%%time
distinct = np.load(REFERENCE / "forest.npz")["distinct_of_hit"]
reference_cluster = np.load(REFERENCE / "probabilities.npy")[distinct].argmax(1)
reference_ids = [f"{reference_hits['sample_ids'][b]}:{t}" for b, t in zip(reference_hits["batch_idx"], reference_hits["token_idx"])]

ours = composition(np.array([group(wnid(h)) for h in space.hit_ids]), space.cluster)
reference = composition(np.array([group(wnid(h)) for h in reference_ids]), reference_cluster)
index = {h: i for i, h in enumerate(space.hit_ids)}
shared = [(index[h], j) for j, h in enumerate(reference_ids) if h in index]
ari = adjusted_rand(space.cluster[[a for a, _ in shared]], reference_cluster[[b for _, b in shared]])

def mixing(rows):
    return [r.cluster for r in rows if min(r.labels.get("cat", 0), r.labels.get("vehicle", 0)) > 0.1 * r.size]

def side(rows, n):
    return {"hits": n, "clusters": len(rows), "weighted_purity": weighted_purity(rows), "mixing_cat_vehicle": mixing(rows),
            "clusters_by_size": [vars(r) for r in rows]}

result = {"shared_hits": len(shared), "ari_on_shared": ari,
          "ours": side(ours, len(space.hit_ids)), "reference": side(reference, len(reference_ids))}
(OUT / "compare.json").write_text(json.dumps(result, indent=1))
for name in ("ours", "reference"):
    r = result[name]
    print(f"{name}: {r['hits']} hits, {r['clusters']} clusters, purity {r['weighted_purity']:.2f}, mixing {r['mixing_cat_vehicle']}")
print("shared", len(shared), "ARI", round(ari, 3))

ours: 13527 hits, 37 clusters, purity 0.82, mixing []
reference: 14240 hits, 41 clusters, purity 0.82, mixing []
shared 13341 ARI 0.605
CPU times: user 6.68 s, sys: 751 ms, total: 7.43 s
Wall time: 7.46 s


## Upload

In [18]:
!tar -cf /content/same_images.tar -C /content/runs same_images
!curl -sSf -X POST -H "Authorization: Bearer $GTOKEN" -H "Content-Type: application/x-tar" --data-binary @/content/same_images.tar "https://storage.googleapis.com/upload/storage/v1/b/pj-okc/o?uploadType=media&name=neural_interp/mixed4e_55/same_images.tar" | head -c 200

{
  "kind": "storage#object",
  "id": "pj-okc/neural_interp/mixed4e_55/same_images.tar/1793038457165352",
  "selfLink": "https://www.googleapis.com/storage/v1/b/pj-okc/o/neural_interp%2Fmixed4e_55%2Fs